In [1]:

"""
This notebook prepares all data needed for the Streamlit interface:
- Patient predictions from all agents
- Fusion predictions
- True labels
- Evidence/explanations
- Interesting case IDs
"""

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

# Disease list
DISEASE_LIST = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE',
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 'ATRIAL_FIBRILLATION',
    'CORONARY_ARTERY_DISEASE', 'ANEMIA', 'PANCREATITIS'
]

print("="*70)
print("📦 PREPARING DATA FOR CONVERSATIONAL INTERFACE")
print("="*70)

📦 PREPARING DATA FOR CONVERSATIONAL INTERFACE


In [2]:


# Load fusion data
X_fusion = np.load('../../data/processed/X_fusion_val.npy')

# Extract agent predictions
agent1_pred = X_fusion[:, 0:9]
agent2_pred = X_fusion[:, 9:18]
agent3_pred = X_fusion[:, 18:27]

# Load labels
y_labels = {}
for disease in DISEASE_LIST:
    disease_filename = disease.lower()
    y_labels[disease] = np.load(f'../../data/processed/y_fusion_val_{disease_filename}.npy')

# Create test split (same as before)
indices = np.arange(len(X_fusion))
train_val_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=42)

print(f"✅ Loaded data")
print(f"   Test set: {len(test_idx)} patients")
print(f"   Each patient has 9 disease predictions")

✅ Loaded data
   Test set: 1393 patients
   Each patient has 9 disease predictions


In [3]:


# Create enhanced features
def create_enhanced_features(X_base):
    n_samples = X_base.shape[0]
    X_enhanced = np.zeros((n_samples, 105))
    X_enhanced[:, 0:33] = X_base
    
    agent1 = X_base[:, 0:9]
    agent2 = X_base[:, 9:18]
    agent3 = X_base[:, 18:27]
    
    idx = 33
    X_enhanced[:, idx:idx+9] = agent1 * agent2; idx += 9
    X_enhanced[:, idx:idx+9] = agent1 * agent3; idx += 9
    X_enhanced[:, idx:idx+9] = agent2 * agent3; idx += 9
    X_enhanced[:, idx:idx+9] = np.abs(agent1 - agent2); idx += 9
    X_enhanced[:, idx:idx+9] = np.abs(agent1 - agent3); idx += 9
    X_enhanced[:, idx:idx+9] = np.abs(agent2 - agent3); idx += 9
    
    agreement = ((agent1 > 0.5) == (agent2 > 0.5)) & ((agent2 > 0.5) == (agent3 > 0.5))
    X_enhanced[:, idx:idx+9] = agreement.astype(float); idx += 9
    
    all_preds = np.stack([agent1, agent2, agent3], axis=0)
    X_enhanced[:, idx:idx+9] = np.var(all_preds, axis=0)
    
    return X_enhanced

X_enhanced = create_enhanced_features(X_fusion)
X_test_enhanced = X_enhanced[test_idx]

# Load thresholds
with open('../../results/optimal_thresholds.json', 'r') as f:
    thresholds = json.load(f)

# Generate fusion predictions
fusion_pred_proba = np.zeros((len(test_idx), 9))
fusion_pred_binary = np.zeros((len(test_idx), 9))

print("Generating fusion predictions for test set...")

for i, disease in enumerate(DISEASE_LIST):
    model_path = f'../../models/fusion/fusion_{disease.lower()}.joblib'
    model = joblib.load(model_path)
    
    fusion_pred_proba[:, i] = model.predict_proba(X_test_enhanced)[:, 1]
    threshold = thresholds.get(disease, 0.5)
    fusion_pred_binary[:, i] = (fusion_pred_proba[:, i] >= threshold).astype(int)

print(f"✅ Generated fusion predictions")

Generating fusion predictions for test set...
✅ Generated fusion predictions


In [4]:


# Extract test set data
agent1_test = agent1_pred[test_idx]
agent2_test = agent2_pred[test_idx]
agent3_test = agent3_pred[test_idx]
y_test = {disease: y_labels[disease][test_idx] for disease in DISEASE_LIST}

# Create patient database
patient_database = []

for idx, patient_id in enumerate(test_idx):
    patient_data = {
        'patient_id': int(patient_id),
        'test_index': idx,  # Index in test arrays
        'true_labels': {},
        'agent1_probs': {},
        'agent2_probs': {},
        'agent3_probs': {},
        'fusion_probs': {},
        'fusion_binary': {},
        'thresholds': {}
    }
    
    # For each disease
    for i, disease in enumerate(DISEASE_LIST):
        patient_data['true_labels'][disease] = int(y_test[disease][idx])
        patient_data['agent1_probs'][disease] = float(agent1_test[idx, i])
        patient_data['agent2_probs'][disease] = float(agent2_test[idx, i])
        patient_data['agent3_probs'][disease] = float(agent3_test[idx, i])
        patient_data['fusion_probs'][disease] = float(fusion_pred_proba[idx, i])
        patient_data['fusion_binary'][disease] = int(fusion_pred_binary[idx, i])
        patient_data['thresholds'][disease] = float(thresholds.get(disease, 0.5))
    
    patient_database.append(patient_data)

print(f"✅ Created patient database: {len(patient_database)} patients")
print(f"   Each patient has predictions for {len(DISEASE_LIST)} diseases")

✅ Created patient database: 1393 patients
   Each patient has predictions for 9 diseases


In [6]:


output_dir = Path('../../interface_data')
output_dir.mkdir(exist_ok=True)

# Save patient database
with open(output_dir / 'patient_database.json', 'w') as f:
    json.dump(patient_database, f, indent=2)

# Save interesting cases
with open(output_dir / 'interesting_cases.json', 'w') as f:
    json.dump(interesting_cases, f, indent=2)

# Save disease list
with open(output_dir / 'disease_list.json', 'w') as f:
    json.dump(DISEASE_LIST, f, indent=2)

# Save thresholds
with open(output_dir / 'thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)

print("\n" + "="*70)
print("✅ ALL DATA SAVED!")
print("="*70)
print(f"\nSaved to: {output_dir.absolute()}")
print(f"\nFiles created:")
print(f"  • patient_database.json ({len(patient_database)} patients)")
print(f"  • interesting_cases.json (5 cases)")
print(f"  • disease_list.json (9 diseases)")
print(f"  • thresholds.json (9 thresholds)")
print("\n🎉 Ready to run Streamlit app!")


✅ ALL DATA SAVED!

Saved to: C:\Users\athar\mimic_project\notebooks\..\interface_data

Files created:
  • patient_database.json (1393 patients)
  • interesting_cases.json (5 cases)
  • disease_list.json (9 diseases)
  • thresholds.json (9 thresholds)

🎉 Ready to run Streamlit app!


In [7]:

print("\n" + "="*70)
print("🔄 UPDATING INTERESTING CASES")
print("="*70)

# Old interesting case indices
old_interesting = {
    'perfect_fusion_win': 4420,
    'high_confidence_fp': 803,
    'narrow_miss': 1149,
    'max_disagreement': 907,
    'perfect_agreement': 4051
}

# Map old indices to actual SUBJECT_IDs
new_interesting = {}

for case_name, old_idx in old_interesting.items():
    # Find this patient in database by test_index
    found = False
    for patient in patient_database:
        # The old "patient_id" was actually the test_idx originally
        # We need to find which patient had test_index close to old_idx
        # Actually, let's just pick representative patients from our test set
        pass
    
# Let's just pick 5 good example patients from our test set
print("Selecting new interesting case examples...")

# Find a perfect fusion win
perfect_win = None
for p in patient_database:
    sepsis_correct = (p['fusion_binary']['SEPSIS'] == p['true_labels']['SEPSIS'])
    agent1_wrong = (p['agent1_probs']['SEPSIS'] >= 0.5) != p['true_labels']['SEPSIS']
    agent2_wrong = (p['agent2_probs']['SEPSIS'] >= 0.5) != p['true_labels']['SEPSIS']
    agent3_wrong = (p['agent3_probs']['SEPSIS'] >= 0.5) != p['true_labels']['SEPSIS']
    
    if sepsis_correct and agent1_wrong and agent2_wrong and agent3_wrong:
        perfect_win = p['patient_id']
        print(f"  Found perfect fusion win: Patient {perfect_win}")
        break

# Find a high confidence FP
high_conf_fp = None
for p in patient_database:
    if p['fusion_probs']['SEPSIS'] > 0.85 and p['true_labels']['SEPSIS'] == 0:
        high_conf_fp = p['patient_id']
        print(f"  Found high confidence FP: Patient {high_conf_fp}")
        break

# Find max disagreement
max_disagree = None
for p in patient_database:
    a1 = p['agent1_probs']['SEPSIS']
    a2 = p['agent2_probs']['SEPSIS']
    a3 = p['agent3_probs']['SEPSIS']
    
    variance = np.var([a1, a2, a3])
    if variance > 0.1:  # High disagreement
        max_disagree = p['patient_id']
        print(f"  Found max disagreement: Patient {max_disagree}")
        break

# Find narrow miss
narrow_miss = None
sepsis_threshold = thresholds.get('SEPSIS', 0.5)
for p in patient_database:
    diff = abs(p['fusion_probs']['SEPSIS'] - sepsis_threshold)
    if diff < 0.05 and p['fusion_binary']['SEPSIS'] != p['true_labels']['SEPSIS']:
        narrow_miss = p['patient_id']
        print(f"  Found narrow miss: Patient {narrow_miss}")
        break

# Find perfect agreement
perfect_agree = None
for p in patient_database:
    correct = sum([
        p['fusion_binary'][d] == p['true_labels'][d] 
        for d in DISEASE_LIST
    ])
    if correct == 9:  # All 9 correct
        perfect_agree = p['patient_id']
        print(f"  Found perfect agreement: Patient {perfect_agree}")
        break

# Use first patient as fallback for any not found
fallback = patient_database[0]['patient_id']

new_interesting_cases = {
    'perfect_fusion_win': perfect_win or fallback,
    'high_confidence_fp': high_conf_fp or fallback,
    'max_disagreement': max_disagree or fallback,
    'narrow_miss': narrow_miss or fallback,
    'perfect_agreement': perfect_agree or fallback
}

# Save updated interesting cases
with open('../../interface_data/interesting_cases.json', 'w') as f:
    json.dump(new_interesting_cases, f, indent=2)

print(f"\n✅ Updated interesting_cases.json with actual SUBJECT_IDs")
print(f"   Cases: {new_interesting_cases}")


🔄 UPDATING INTERESTING CASES
Selecting new interesting case examples...
  Found perfect fusion win: Patient 4420
  Found high confidence FP: Patient 803
  Found max disagreement: Patient 2277
  Found narrow miss: Patient 1149
  Found perfect agreement: Patient 2707

✅ Updated interesting_cases.json with actual SUBJECT_IDs
   Cases: {'perfect_fusion_win': 4420, 'high_confidence_fp': 803, 'max_disagreement': 2277, 'narrow_miss': 1149, 'perfect_agreement': 2707}


In [8]:
# Check if our interesting cases are actually in the database
print("Verifying interesting cases are in patient_database...")

for case_name, subject_id in new_interesting_cases.items():
    found = any(p['patient_id'] == subject_id for p in patient_database)
    print(f"  {case_name} (Patient {subject_id}): {'✅ Found' if found else '❌ NOT FOUND'}")

# Show all patient IDs in database
all_patient_ids = [p['patient_id'] for p in patient_database]
print(f"\nTotal patients in database: {len(all_patient_ids)}")
print(f"Patient ID range: {min(all_patient_ids)} to {max(all_patient_ids)}")

# Check if 803 specifically is there
if 803 in all_patient_ids:
    print(f"\n✅ Patient 803 IS in database")
else:
    print(f"\n❌ Patient 803 NOT in database")
    print(f"   Closest patients to 803: {sorted([p for p in all_patient_ids if abs(p - 803) < 100])[:5]}")

Verifying interesting cases are in patient_database...
  perfect_fusion_win (Patient 4420): ✅ Found
  high_confidence_fp (Patient 803): ✅ Found
  max_disagreement (Patient 2277): ✅ Found
  narrow_miss (Patient 1149): ✅ Found
  perfect_agreement (Patient 2707): ✅ Found

Total patients in database: 1393
Patient ID range: 6 to 4642

✅ Patient 803 IS in database


In [10]:

print("🔄 Re-saving patient database to ensure sync...")

# Verify 803 is in current patient_database in memory
patient_ids_in_memory = [p['patient_id'] for p in patient_database]
print(f"Patient 803 in memory: {803 in patient_ids_in_memory}")
print(f"Total patients in memory: {len(patient_ids_in_memory)}")
print(f"Patient ID range in memory: {min(patient_ids_in_memory)} to {max(patient_ids_in_memory)}")

# Save again
output_path = '../../interface_data/patient_database.json'
with open(output_path, 'w') as f:
    json.dump(patient_database, f, indent=2)

print(f"\n✅ Re-saved to: {output_path}")

# Verify it saved correctly
with open(output_path, 'r') as f:
    reloaded = json.load(f)

reloaded_ids = [p['patient_id'] for p in reloaded]
print(f"\nVerification after save:")
print(f"  Patient 803 in saved file: {803 in reloaded_ids}")
print(f"  Total patients in file: {len(reloaded_ids)}")
print(f"  Patient ID range: {min(reloaded_ids)} to {max(reloaded_ids)}")

if 803 not in reloaded_ids:
    print(f"\n⚠️ Patient 803 is NOT in the file!")
    print(f"   The interesting case SUBJECT_IDs don't match test set")
    print(f"\n💡 We need to pick interesting cases from ACTUAL test patients")

🔄 Re-saving patient database to ensure sync...
Patient 803 in memory: True
Total patients in memory: 1393
Patient ID range in memory: 6 to 4642

✅ Re-saved to: ../interface_data/patient_database.json

Verification after save:
  Patient 803 in saved file: True
  Total patients in file: 1393
  Patient ID range: 6 to 4642
